# RetailSense AI — Phase 4 : Machine Learning

**Projet d'intégration — Technicien en Intelligence Artificielle · College CDI**
Encadrant : Reda Mohammed Chatou

Ce notebook met en pratique le **Chapitre 6** de la formation sur le jeu de données **Olist**, à partir
de la base **SQLite** construite en phase 1. Il couvre trois sous-projets :

1. **Segmentation client** (non supervisé) — K-means et DBSCAN
2. **Classification supervisée** — régression logistique, arbre de décision, Random Forest, AdaBoost, Gradient Boosting
3. **Régression** — régression linéaire (référence) puis modèles d'ensemble

Chaque modèle est accompagné d'une **justification des paramètres et hyperparamètres**. Les métriques
sont choisies en fonction du besoin métier (et non par défaut).

> **Prérequis** : `pip install pandas numpy scikit-learn matplotlib joblib`
> **Base** : indiquez le chemin de `olist.sqlite` ci-dessous (variable `DB_PATH`).


In [6]:
from sklearn.utils.class_weight import compute_sample_weight
import joblib
from pathlib import Path

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Détection automatique de la racine du projet pour résoudre le bon chemin DB.
ROOT_DIR = None
cwd = Path.cwd().resolve()
for candidate in [cwd, *cwd.parents]:
    if (candidate / 'db').exists() and (candidate / 'data').exists():
        ROOT_DIR = candidate
        break
if ROOT_DIR is None:
    raise FileNotFoundError("Impossible de trouver la racine du projet avec les dossiers db/ et data/.")
DEFAULT_DB = ROOT_DIR / 'db' / 'retailsense.db'
DB_PATH = Path(os.environ.get("OLIST_DB", str(DEFAULT_DB)))
if not DB_PATH.exists():
    raise FileNotFoundError(f"Base SQLite introuvable : {DB_PATH}")
os.makedirs("models", exist_ok=True)
print("Base utilisée :", DB_PATH)
conn = sqlite3.connect(str(DB_PATH))
print("Tables :", [r[0] for r in conn.execute(
    "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'").fetchall()])


Base utilisée : E:\Data\CDI_College\Cours_Profession_de_Inteligence_Artificiel\16- Projet_intérgration\RetailSenseAI\db\retailsense.db
Tables : ['product_category_name_translation', 'geolocation', 'customers', 'sellers', 'products', 'orders', 'order_items', 'payments', 'reviews']


## 1. Construction des variables (feature engineering)

On part de la base relationnelle pour construire une **table au niveau commande** : on agrège les
articles, les paiements et l'avis de chaque commande, puis on y ajoute des variables temporelles et
la localisation du client. C'est la matière première des modèles de classification et de régression.

Les transformations sont volontairement explicites (et reproductibles) — c'est exactement le travail
attendu en phase 2.


In [9]:
# --- Chargement des tables --------------------------------------------
items = pd.read_sql("SELECT * FROM order_items", conn)
prods = pd.read_sql("SELECT product_id, product_category_name, product_weight_g FROM products", conn)
trans = pd.read_sql("SELECT * FROM product_category_name_translation", conn)
pays  = pd.read_sql("SELECT * FROM payments", conn)
revs  = pd.read_sql("SELECT order_id, review_score FROM reviews", conn)
orders = pd.read_sql("SELECT * FROM orders", conn)
cust   = pd.read_sql("SELECT customer_id, customer_unique_id, customer_state FROM customers", conn)

# Catégorie en anglais (sinon nom original, sinon 'unknown')
prods = prods.merge(trans, on="product_category_name", how="left")
prods["category"] = (prods["product_category_name_english"]
                     .fillna(prods["product_category_name"]).fillna("unknown"))
items = items.merge(prods[["product_id", "category", "product_weight_g"]],
                    on="product_id", how="left")

def mode_or_unknown(s):
    s = s.dropna()
    return s.mode().iat[0] if not s.mode().empty else "unknown"

# --- Agrégations au niveau commande -----------------------------------
items_agg = items.groupby("order_id").agg(
    n_items=("order_item_id", "count"),
    total_price=("price", "sum"),
    total_freight=("freight_value", "sum"),
    total_weight=("product_weight_g", "sum"),
    main_category=("category", mode_or_unknown)).reset_index()

pays_agg = pays.groupby("order_id").agg(
    payment_value=("payment_value", "sum"),
    max_installments=("payment_installments", "max"),
    payment_type=("payment_type", mode_or_unknown)).reset_index()

revs_agg = revs.groupby("order_id").agg(review_score=("review_score", "min")).reset_index()

# --- Variables temporelles / livraison --------------------------------
orders = orders.merge(cust, on="customer_id", how="left")
for c in ["order_purchase_timestamp", "order_delivered_customer_date",
          "order_estimated_delivery_date"]:
    orders[c] = pd.to_datetime(orders[c], errors="coerce")
orders["delivery_days"] = (orders["order_delivered_customer_date"]
                           - orders["order_purchase_timestamp"]).dt.days
orders["delay_days"] = (orders["order_delivered_customer_date"]
                        - orders["order_estimated_delivery_date"]).dt.days
orders["purchase_month"] = orders["order_purchase_timestamp"].dt.month
orders["purchase_dow"]   = orders["order_purchase_timestamp"].dt.dayofweek

df = (orders.merge(items_agg, on="order_id", how="left")
            .merge(pays_agg, on="order_id", how="left")
            .merge(revs_agg, on="order_id", how="left"))
df["bad_review"] = np.where(df["review_score"].notna(),
                            (df["review_score"] <= 2).astype(float), np.nan)
print("Table au niveau commande :", df.shape)
df.head(3)


Table au niveau commande : (99441, 24)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_state,...,n_items,total_price,total_freight,total_weight,main_category,payment_value,max_installments,payment_type,review_score,bad_review
0,00010242fe8c5a6d1ba2dd792cb16214,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29,871766c5855e863f6eccc05f988b23cb,RJ,...,1.0,58.9,13.29,650.0,cool_stuff,72.19,2.0,credit_card,5.0,0.0
1,00018f77f2f0320c557190d7a144bdd3,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15,eb28e67c4c0b83846050ddfb8a35d051,SP,...,1.0,239.9,19.93,30000.0,pet_shop,259.83,3.0,credit_card,4.0,0.0
2,000229ec398224ef6ca0657da4fc703e,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05,3818d81c6709e39d06b2738a8d3a2474,MG,...,1.0,199.0,17.87,3050.0,furniture_decor,216.87,5.0,credit_card,5.0,0.0


## 2. Segmentation de la clientèle (apprentissage non supervisé)

**Objectif métier** : regrouper les clients pour adapter le marketing (clients fidèles à forte valeur,
clients dormants, etc.). On utilise la **matrice RFM** (Récence, Fréquence, Montant) calculée par
**personne réelle** (`customer_unique_id`, et non `customer_id`).


In [10]:
# --- Matrice RFM par client (personne réelle) -------------------------
order_value = df[["order_id", "payment_value", "customer_unique_id",
                  "order_purchase_timestamp"]].copy()
ref_date = order_value["order_purchase_timestamp"].max()

rfm = order_value.groupby("customer_unique_id").agg(
    recency=("order_purchase_timestamp", lambda s: (ref_date - s.max()).days),
    frequency=("order_id", "nunique"),
    monetary=("payment_value", "sum")).reset_index()
rfm = rfm.dropna(subset=["monetary"])
print("Clients :", len(rfm))
rfm.describe()[["recency", "frequency", "monetary"]]


Clients : 96096


,recency,frequency,monetary
count,96096.000000,96096.000000,96096.000000
mean,287.735691,1.034809,166.592492
std,153.414676,0.214384,231.428332
min,0.000000,1.000000,0.000000
25%,163.000000,1.000000,63.120000
50%,268.000000,1.000000,108.000000
75%,397.000000,1.000000,183.530000
max,772.000000,17.000000,13664.080000


In [11]:
# --- K-means : choix de k (coude + silhouette) ------------------------
# La distance euclidienne impose de STANDARDISER les variables (sinon le
# 'monetary', d'amplitude bien plus grande, domine la distance).
X_rfm = rfm[["recency", "frequency", "monetary"]].copy()
# Log sur monetary/frequency pour atténuer l'asymétrie, puis standardisation
X_rfm["monetary"]  = np.log1p(X_rfm["monetary"])
X_rfm["frequency"] = np.log1p(X_rfm["frequency"])
scaler_rfm = StandardScaler()
Xs = scaler_rfm.fit_transform(X_rfm)